# EDA dan pemeriksaan data — Kasus 03
Notebook ini membaca data mentah, memeriksa kualitasnya, lalu membuat grafik hanya dari train + validation. Data test tetap ditahan untuk evaluasi final. Jalankan dari folder utama setelah `python -m pip install -r requirements-dev.txt`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
df_raw = pd.read_csv(ROOT / 'data/raw/raw.csv')
split = pd.read_csv(ROOT / 'artifacts/split_motor_kasus03.csv')
FIG_DIR = ROOT / 'reports/figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('Ukuran data mentah:', df_raw.shape)
display(df_raw.head())

## Pemeriksaan dan pembersihan
ID motor dinormalisasi sebelum split diterapkan. Baris duplikat identik dihapus dari salinan analisis, bukan dari CSV sumber.

In [ ]:
df = df_raw.copy()
df['motor_id'] = df['motor_id'].astype(str).str.replace(r'[^A-Za-z0-9]', '', regex=True).str.upper()
df['tipe_motor'] = (df['tipe_motor'].astype(str).str.strip().str.lower().str.replace(r'\s+', '', regex=True).str.replace(',', '.', regex=False))
type_map = {'5.5kw':'5.5kW', '11kw':'11kW', '22kw':'22kW'}
df['tipe_motor'] = df['tipe_motor'].map(type_map).fillna('UNKNOWN')
duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
numeric = ['hari_operasi','beban_pct','arus_A','tegangan_V','suhu_belitan_C','getaran_rms_mm_s','kecepatan_rpm','rul_hari']
for col in numeric: df[col] = pd.to_numeric(df[col], errors='coerce')
negative_vibration = int((df['getaran_rms_mm_s'] < 0).sum())
missing = df[numeric].isna().sum()
df.loc[df['getaran_rms_mm_s'] < 0, 'getaran_rms_mm_s'] = pd.NA
print('Duplikat identik:', duplicate_count, '| Motor:', df.motor_id.nunique(), '| Getaran negatif:', negative_vibration)
display(missing.to_frame('nilai kosong sebelum koreksi RMS'))
display(df[numeric].describe().T)

## Pembagian per motor dan EDA
Manifest memastikan semua catatan satu motor berada di bagian yang sama. Test tidak dipakai dalam eksplorasi atau pemilihan model.

In [ ]:
df = df.merge(split, on='motor_id', how='left', validate='many_to_one')
assert df.split.notna().all()
assert split.motor_id.is_unique
display(df.groupby('split').agg(baris=('rul_hari','size'), motor=('motor_id','nunique')))
eda = df[df.split.isin(['train','validation'])].copy()
bins = [-1,30,60,90,120,180,240,float('inf')]
labels = ['0–30','31–60','61–90','91–120','121–180','181–240','241+']
eda['rentang_rul'] = pd.cut(eda.rul_hari, bins=bins, labels=labels)
counts = eda.rentang_rul.value_counts(sort=False)
display(counts.to_frame('jumlah baris'))
ax = counts.plot(kind='bar', figsize=(9,4), title='Distribusi RUL — train + validation', color='#2878B5')
ax.set_xlabel('Rentang RUL (hari)'); ax.set_ylabel('Jumlah baris')
plt.tight_layout(); plt.savefig(FIG_DIR / 'distribusi_rul.png', dpi=160); plt.show()

In [ ]:
sensors = ['beban_pct','arus_A','suhu_belitan_C','getaran_rms_mm_s']
medians = eda.groupby('rentang_rul', observed=False)[sensors].median()
display(medians.round(2))
ax = medians[['arus_A','suhu_belitan_C','getaran_rms_mm_s']].plot(marker='o', figsize=(10,5), title='Median sensor menurut rentang RUL')
ax.set_xlabel('Rentang RUL (hari)'); ax.set_ylabel('Nilai sensor')
plt.tight_layout(); plt.savefig(FIG_DIR / 'median_sensor_per_rul.png', dpi=160); plt.show()

## Hasil model
Validation dipakai memilih model. Setelah model dikunci, skrip `src/train_final.py` melatih ulang pada train + validation dan mengevaluasi test satu kali. Jangan gunakan test untuk mengubah model.

In [ ]:
with open(ROOT/'artifacts/baseline_linear_kasus03.json', encoding='utf-8-sig') as f: baseline = json.load(f)
with open(ROOT/'artifacts/polynomial_regression_kasus03.json', encoding='utf-8') as f: candidate = json.load(f)
with open(ROOT/'artifacts/final_metrics.json', encoding='utf-8') as f: final = json.load(f)
display(pd.DataFrame([{'model':'Baseline linear',**baseline['metrics_validation']},{'model':'Polinomial derajat 2',**candidate['metrics_validation']}]))
display(pd.DataFrame([{'model':'Baseline linear',**final['baseline_test']},{'model':'Polinomial derajat 2',**final['final_model_test']}]))